# Tarea 1 – Parte 2: Del LLM al clasificador

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

Entrada: `datos/noticias.csv` (Parte 1). La columna `etiqueta` (derivada de la sección del medio) se usa como **referencia** para evaluar los tres enfoques:

1. **Zero-shot** (sin entrenamiento) → piso mínimo.
2. **LLM como supervisión débil** → `datos/etiquetas_llm.csv` + auditoría humana de 20 casos.
3. **Transformer fine-tuned** (`distilbert-base-multilingual-cased`) entrenado con el split de train.

> El conjunto de **test** se separa al inicio con `random_state=42` y no se usa hasta la evaluación final (sección 5). Todo se evalúa solo con el `titular`, que es lo que recibirá el dashboard.

In [ ]:
import json, os, re, time
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
DATOS, MODELO, GRAFICOS = Path("datos"), Path("modelo"), Path("graficos")
MODELO.mkdir(exist_ok=True); GRAFICOS.mkdir(exist_ok=True)
METRICAS = MODELO / "metricas.json"

def guardar_metricas(**kw):
    m = json.loads(METRICAS.read_text()) if METRICAS.exists() else {}
    m.update(kw); METRICAS.write_text(json.dumps(m, ensure_ascii=False, indent=2))

df = pd.read_csv(DATOS / "noticias.csv")
print(df.shape); df.etiqueta.value_counts()

## 0. Partición estratificada 70 / 15 / 15 (`random_state=42`)

In [ ]:
train, temp = train_test_split(df, test_size=0.30, stratify=df.etiqueta, random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, stratify=temp.etiqueta, random_state=SEED)
for nombre, parte in [("train", train), ("val", val), ("test", test)]:
    print(f"{nombre:5s}: {len(parte):3d} filas ({len(parte)/len(df):.0%}) | {parte.etiqueta.value_counts().to_dict()}")
(MODELO / "split_urls.json").write_text(json.dumps({"train": list(train.url), "val": list(val.url), "test": list(test.url)}, indent=1))
assert not (set(train.url) & set(test.url)) and not (set(val.url) & set(test.url))

## 1. Línea base zero-shot (sin fine-tuning)
40 titulares del **test** (10 por categoría, muestreo con semilla fija) con `Recognai/bert-base-spanish-wwm-cased-xnli`. Las etiquetas se pasan en lenguaje natural.

In [ ]:
from transformers import pipeline

NOMBRES = {"politica": "política", "economia": "economía", "deportes": "deportes", "ciencia_tecnologia": "ciencia y tecnología"}
INVERSO = {v: k for k, v in NOMBRES.items()}
zs = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli")

muestra_zs = test.groupby("etiqueta", group_keys=False).apply(lambda g: g.sample(min(10, len(g)), random_state=SEED))
t0 = time.time()
salidas = zs(list(muestra_zs.titular), candidate_labels=list(NOMBRES.values()), hypothesis_template="Esta noticia trata sobre {}.")
lat_zs = (time.time() - t0) / len(muestra_zs)
muestra_zs["pred_zs"] = [INVERSO[s["labels"][0]] for s in salidas]
acc_zs = accuracy_score(muestra_zs.etiqueta, muestra_zs.pred_zs)
f1_zs = f1_score(muestra_zs.etiqueta, muestra_zs.pred_zs, average="macro")
print(f"Zero-shot ({len(muestra_zs)} titulares) → accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f} | latencia: {lat_zs*1000:.0f} ms/titular")
guardar_metricas(zero_shot_accuracy=round(acc_zs, 4), zero_shot_f1_macro=round(f1_zs, 4), zero_shot_n=len(muestra_zs))
print(classification_report(muestra_zs.etiqueta, muestra_zs.pred_zs, zero_division=0))

## 2. Supervisión débil con LLM
Prompt estructurado que obliga a responder JSON con una de las 4 categorías y una confianza 0–1. Backends soportados:
- **Gemini API** (`LLM_BACKEND=gemini`, requiere `GEMINI_API_KEY` en un archivo `.env` que **no** se sube a GitHub).
- **Ollama local** (`LLM_BACKEND=ollama`, p. ej. `qwen2.5:7b`).

Las respuestas se guardan de forma incremental en `datos/etiquetas_llm.csv` (si se interrumpe, se reanuda).

In [ ]:
if Path(".env").exists():
    for linea in Path(".env").read_text().splitlines():
        if "=" in linea and not linea.startswith("#"):
            k, v = linea.split("=", 1); os.environ.setdefault(k.strip(), v.strip())

LLM_BACKEND = os.environ.get("LLM_BACKEND", "gemini")
LLM_MODELO = os.environ.get("LLM_MODELO", "gemini-2.5-flash" if LLM_BACKEND == "gemini" else "qwen2.5:7b")
PROMPT = """Eres un editor de un diario peruano. Clasifica el titular en UNA de estas categorías:
- politica: gobierno, Congreso, partidos, elecciones, justicia y corrupción de funcionarios, política internacional.
- economia: mercados, empresas, empleo, precios, impuestos, AFP, finanzas personales, comercio.
- deportes: fútbol y cualquier otra disciplina deportiva, clubes, selecciones, resultados.
- ciencia_tecnologia: ciencia, salud científica, espacio, medio ambiente científico, tecnología, internet, apps, IA, videojuegos.
Responde SOLO con JSON: {{"etiqueta": "<politica|economia|deportes|ciencia_tecnologia>", "confianza": <número entre 0 y 1>}}
Titular: "{titular}" """

def consultar_llm(titular):
    texto = PROMPT.format(titular=titular)
    if LLM_BACKEND == "gemini":
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODELO}:generateContent"
        r = requests.post(url, params={"key": os.environ["GEMINI_API_KEY"]}, timeout=60, json={
            "contents": [{"parts": [{"text": texto}]}],
            "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}})
        r.raise_for_status()
        salida = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    else:
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": LLM_MODELO, "prompt": texto, "format": "json", "stream": False, "options": {"temperature": 0}})
        r.raise_for_status()
        salida = r.json()["response"]
    datos = json.loads(re.search(r"\{.*\}", salida, re.S).group(0))
    etiqueta = str(datos.get("etiqueta", "")).strip().lower()
    return (etiqueta if etiqueta in CATEGORIAS else "invalida"), float(datos.get("confianza", np.nan))

RUTA_LLM = DATOS / "etiquetas_llm.csv"
previas = pd.read_csv(RUTA_LLM) if RUTA_LLM.exists() else pd.DataFrame(columns=["titular", "etiqueta_llm", "confianza", "modelo_llm", "fecha"])
hechos = set(previas.titular)
nuevas, t0, n0 = [], time.time(), 0
for titular in df.titular:
    if titular in hechos:
        continue
    for intento in range(2):
        try:
            etiqueta, conf = consultar_llm(titular); break
        except Exception as e:
            etiqueta, conf = "error", np.nan; time.sleep(2)
    nuevas.append({"titular": titular, "etiqueta_llm": etiqueta, "confianza": conf, "modelo_llm": f"{LLM_BACKEND}:{LLM_MODELO}", "fecha": date.today().isoformat()})
    n0 += 1
    if len(nuevas) % 25 == 0:
        pd.concat([previas, pd.DataFrame(nuevas)]).to_csv(RUTA_LLM, index=False)
    time.sleep(0.5)
etiquetas_llm = pd.concat([previas, pd.DataFrame(nuevas)], ignore_index=True)
etiquetas_llm.to_csv(RUTA_LLM, index=False)
lat_llm = (time.time() - t0) / n0 if n0 else np.nan
print(etiquetas_llm.shape, etiquetas_llm.etiqueta_llm.value_counts().to_dict())
etiquetas_llm.head()

### 2.1 Auditoría humana de 20 predicciones al azar
Se genera `datos/auditoria_llm_20.csv` con 20 titulares al azar (`random_state=42`). **La persona que audita** lee cada titular (y su bajada/URL) y escribe en `etiqueta_humana` la categoría correcta según su criterio. Luego se reejecuta esta celda para calcular el acuerdo.

In [ ]:
RUTA_AUD = DATOS / "auditoria_llm_20.csv"
if not RUTA_AUD.exists():
    aud = (etiquetas_llm.merge(df[["titular", "bajada", "url"]], on="titular", how="left")
           .sample(20, random_state=SEED)[["titular", "bajada", "url", "etiqueta_llm", "confianza"]])
    aud["etiqueta_humana"] = ""
    aud.to_csv(RUTA_AUD, index=False)
aud = pd.read_csv(RUTA_AUD).fillna({"etiqueta_humana": ""})
completa = aud.etiqueta_humana.isin(CATEGORIAS).all()
if completa:
    aciertos = int((aud.etiqueta_humana == aud.etiqueta_llm).sum())
    print(f"Acuerdo LLM vs. auditoría humana: {aciertos}/20 = {aciertos/20:.0%}")
    guardar_metricas(acuerdo_llm_humano=aciertos, auditoria_n=20)
    display(aud[aud.etiqueta_humana != aud.etiqueta_llm])
else:
    print(f"PENDIENTE: completar 'etiqueta_humana' en {RUTA_AUD} ({(~aud.etiqueta_humana.isin(CATEGORIAS)).sum()} filas sin revisar)")
aud

## 3. Fine-tuning de `distilbert-base-multilingual-cased`
HuggingFace `Trainer`: 3 épocas, batch 16, evaluación por época sobre **val**, mejor modelo según **F1 macro**. Se entrena con `etiqueta` (referencia por sección). Los pesos se guardan en `modelo/` y están excluidos de git (`.gitignore`) por superar los 100 MB.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed

set_seed(SEED)
BASE = "distilbert-base-multilingual-cased"
id2label = dict(enumerate(CATEGORIAS)); label2id = {v: k for k, v in id2label.items()}
tok = AutoTokenizer.from_pretrained(BASE)
modelo = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=4, id2label=id2label, label2id=label2id)

class Titulares(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tok(list(d.titular), truncation=True, max_length=64, padding="max_length")
        self.y = [label2id[e] for e in d.etiqueta]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i]); return item

def metricas(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pred), "f1_macro": f1_score(p.label_ids, pred, average="macro")}

args = TrainingArguments(output_dir=str(MODELO / "entrenamiento"), num_train_epochs=3, per_device_train_batch_size=16,
                         per_device_eval_batch_size=32, learning_rate=5e-5, weight_decay=0.01, eval_strategy="epoch",
                         save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1_macro",
                         save_total_limit=1, logging_strategy="epoch", seed=SEED, report_to="none")
trainer = Trainer(model=modelo, args=args, train_dataset=Titulares(train), eval_dataset=Titulares(val), compute_metrics=metricas)
t0 = time.time(); trainer.train(); print(f"Entrenamiento: {time.time()-t0:.0f} s")
pd.DataFrame([l for l in trainer.state.log_history if "eval_f1_macro" in l])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]

In [ ]:
trainer.save_model(str(MODELO)); tok.save_pretrained(str(MODELO))
print(sorted(p.name for p in MODELO.iterdir()))

## 4. Evaluación final en el test reservado

In [ ]:
t0 = time.time()
pred = trainer.predict(Titulares(test))
lat_ft = (time.time() - t0) / len(test)
y_true = [id2label[i] for i in pred.label_ids]; y_pred = [id2label[i] for i in pred.predictions.argmax(-1)]
acc_ft, f1_ft = accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")
print(f"TEST ({len(test)} titulares) → accuracy: {acc_ft:.3f} | F1 macro: {f1_ft:.3f}")
print(classification_report(y_true, y_pred, digits=3))
guardar_metricas(test_accuracy=round(acc_ft, 4), test_f1_macro=round(f1_ft, 4), test_n=len(test), modelo_base=BASE)

cm = confusion_matrix(y_true, y_pred, labels=CATEGORIAS)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIAS, yticklabels=CATEGORIAS)
plt.xlabel("Predicción"); plt.ylabel("Etiqueta real"); plt.title(f"Matriz de confusión en test (F1 macro = {f1_ft:.2f})")
plt.tight_layout(); plt.savefig(GRAFICOS / "04_matriz_confusion_test.png", dpi=150); plt.show()

### 4.1 Reflexión por categoría
_(Completar después de ejecutar con los valores de `classification_report`: categoría con mejor y peor F1 y su causa lingüística o temática, p. ej. solapamiento entre `economia` y `politica` en titulares sobre presupuesto, impuestos o AFP.)_

## 5. Comparativa y decisión

In [ ]:
test_llm = test.merge(etiquetas_llm[["titular", "etiqueta_llm"]], on="titular", how="left")
acc_llm = accuracy_score(test_llm.etiqueta, test_llm.etiqueta_llm)
f1_llm = f1_score(test_llm.etiqueta, test_llm.etiqueta_llm, average="macro", labels=CATEGORIAS)
guardar_metricas(llm_test_accuracy=round(acc_llm, 4), llm_test_f1_macro=round(f1_llm, 4), llm_modelo=f"{LLM_BACKEND}:{LLM_MODELO}")
comparativa = pd.DataFrame([
    {"enfoque": "Zero-shot (bert-base-spanish-wwm-cased-xnli)", "evaluado_en": f"{len(muestra_zs)} titulares de test", "accuracy": acc_zs, "f1_macro": f1_zs, "latencia_ms": lat_zs * 1000, "costo": "CPU, sin entrenamiento"},
    {"enfoque": f"LLM débil ({LLM_BACKEND}:{LLM_MODELO})", "evaluado_en": f"{len(test_llm)} titulares de test", "accuracy": acc_llm, "f1_macro": f1_llm, "latencia_ms": lat_llm * 1000, "costo": "API/GPU por consulta"},
    {"enfoque": "DistilBERT multilingüe fine-tuned", "evaluado_en": f"{len(test)} titulares de test", "accuracy": acc_ft, "f1_macro": f1_ft, "latencia_ms": lat_ft * 1000, "costo": "Entrenamiento único en CPU; inferencia local"},
]).round(3)
comparativa

### 5.1 Justificación
_(Completar después de ejecutar: elegir el modelo para un entorno real ponderando F1 macro en test, costo por predicción y latencia de la tabla anterior.)_